In [3]:
from google.colab import userdata
from huggingface_hub import InferenceClient

HF_TOKEN = userdata.get("HF_TOKEN")

client = InferenceClient(
    model="moonshotai/Kimi-K2.5",
    token=HF_TOKEN
)

In [1]:
!pip install -U huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 839.9/839.9 kB 3.5 MB/s eta 0:00:00
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.33.0
    Uninstalling huggingface_hub-1.33.0:
      Successfully uninstalled huggingface_hub-1.33.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 2.2.0 which is incompatible.
tokenizers 0.23.2 requires huggingface-hub<2.0,>=0.16.4, but you have huggingface-hub 2.2.0 which is incompatible.
gradio-client 2.7.1 requires huggingface-hub<2.0,>=0.19.3, but you have huggingface-hub 2.2.0 which is incompatible.
datasets 4.8.5 requires huggingface-hub<2.0,>=0.25.0, but you have huggingface-hub 2.2.0 which is incompatible.


In [4]:
from google.colab import userdata
from huggingface_hub import InferenceClient

HF_TOKEN = userdata.get("HF_TOKEN")

client = InferenceClient(
    model="moonshotai/Kimi-K2.5",
    token=HF_TOKEN
)

In [5]:
SYSTEM_PROMPT = """
You are a simple AI agent.

You have access to this tool:

get_weather: Get weather information
for a given location.

Use this JSON format for actions:
{
    "action": "get_weather",
    "action_input": {
        "location": "London"
    }
}

Follow this format:

Thought: Explain the next step briefly.
Action: Request one tool action.
Observation: Use the actual tool result.
Final Answer: Provide the final response.

Never invent an observation.
"""

In [10]:

import requests

def get_weather(location="Kolkata"):
    # Coordinates for Kolkata
    locations = {
        "kolkata": (22.5726, 88.3639),
        "delhi": (28.6139, 77.2090),
        "mumbai": (19.0760, 72.8777),
        "chennai": (13.0827, 80.2707)
    }

    city = location.strip().lower()

    if city not in locations:
        return f"Location not configured: {location}"

    lat, lon = locations[city]

    url = "https://api.open-meteo.com/v1/forecast"

    params = {
        "latitude": lat,
        "longitude": lon,
        "current": (
            "temperature_2m,"
            "relative_humidity_2m,"
            "wind_speed_10m"
        )
    }

    try:
        response = requests.get(
            url, params=params, timeout=20
        )
        response.raise_for_status()

        current = response.json()["current"]

        return (
            f"Weather in {location.title()}:\n"
            f"Temperature: {current['temperature_2m']} °C\n"
            f"Humidity: {current['relative_humidity_2m']}%\n"
            f"Wind speed: {current['wind_speed_10m']} km/h"
        )

    except requests.RequestException as e:
        return f"Weather API error: {e}"


print(get_weather("Kolkata"))


Weather in Kolkata:
Temperature: 32.5 °C
Humidity: 53%
Wind speed: 8.4 km/h


In [12]:

def dummy_agent(question, location):
    print("Question:", question)

    # 1. Thought
    print("\nThought: I need current weather information.")

    # 2. Action
    print("Action: get_weather")
    print("Action Input:", location)

    # 3. Execute the tool and get observation
    result = get_weather(location)

    print("\nObservation:")
    print(result)

    # 4. Final Answer
    print("\nThought: I now have the weather information.")
    print("Final Answer:", result)


dummy_agent(
    "What's the weather in Kolkata?",
    "Kolkata"
)


Question: What's the weather in Kolkata?

Thought: I need current weather information.
Action: get_weather
Action Input: Kolkata

Observation:
Weather in Kolkata:
Temperature: 32.5 °C
Humidity: 53%
Wind speed: 8.4 km/h

Thought: I now have the weather information.
Final Answer: Weather in Kolkata:
Temperature: 32.5 °C
Humidity: 53%
Wind speed: 8.4 km/h
